# map.art — build training set v01

Production data-build notebook. Single purpose: take every render tile under `data/renderer/samples/`, stylize it individually via OpenAI **gpt-image-2** at 1024×1024, and write aligned `(source.png, target.png, meta.json)` triples into `python/training/v01/{scene_id}/{tile_id}/`.

Decisions baked in (do not re-litigate here — see `exploration.ipynb` for the comparisons):

- **Model:** gpt-image-2 (no fidelity flag — it rejects the param).
- **Mode:** single tile → single output. No stitching, no mask. The model fine-tune we train downstream will handle seams/infill via Andy Coenen's quadrant-mask augmentation — we just need clean base pairs here.
- **Resolution:** 512×512 renderer input → 1024×1024 stylized output. The model implicitly learns a 2× upscale that becomes useful at inference time.
- **Resumable:** if `target.png` exists, the tile is skipped. Re-run the batch cell to fill in failures.
- **Prompt:** unchanged from `exploration.ipynb` cell 13 — same SimCity 2000 / RCT2 / Theme Hospital direction.

After this notebook completes, the next steps live in **future** notebooks: visual cull → mask augmentation (Andy's 8 variants per pair) → horizontal flip → oxen.ai manifest CSV → Qwen Image Edit LoRA fine-tune.

## 1. Setup

In [2]:
!pip install python-dotenv openai pillow matplotlib
# Optional (Imagen via Vertex AI):
!pip install google-cloud-aiplatform

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached openai-2.37.0-py3-none-any.whl.metadata (31 kB)
  Using cached pillow-12.2.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached matplotlib-3.10.9-cp314-cp314-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.15-py3-none-any.whl.metadata (7.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata 

In [3]:
import base64
import io
import json
import os
import time
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from PIL import Image


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(10):
        if (cur / 'pnpm-workspace.yaml').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('repo root not found')


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
load_dotenv(REPO_ROOT / '.env')

TRAINING_VERSION = 'v01'
SAMPLES_ROOT = REPO_ROOT / 'data' / 'renderer' / 'samples'
TRAINING_ROOT = REPO_ROOT / 'python' / 'training' / TRAINING_VERSION
TRAINING_ROOT.mkdir(parents=True, exist_ok=True)

print('repo root      :', REPO_ROOT)
print('samples root   :', SAMPLES_ROOT.relative_to(REPO_ROOT))
print('training root  :', TRAINING_ROOT.relative_to(REPO_ROOT))
print('OPENAI_API_KEY :', 'set' if os.environ.get('OPENAI_API_KEY') else 'MISSING')

repo root      : /Users/tiagotrindade/Desktop/MapArt/map.art
samples root   : data/renderer/samples
training root  : python/training/v01
OPENAI_API_KEY : set


## 2. Discover every rendered tile

Walks `data/renderer/samples/<run>/sample<i>/c<col>_r<row>.png` and builds the work list. Each tile becomes one training pair (one API call).

In [4]:
def discover_tiles(root: Path) -> list[dict]:
    """Return [{run, sample, col, row, src_path, scene_id, tile_id}, ...] for every tile.

    scene_id = '<run>__<sample>' — unique per 3×3 cluster.
    tile_id  = 'c{col}_r{row}'   — unique within a scene.
    """
    work: list[dict] = []
    if not root.is_dir():
        return work
    for run_dir in sorted(root.iterdir()):
        if not run_dir.is_dir():
            continue
        for sample_dir in sorted(run_dir.iterdir()):
            if not sample_dir.is_dir():
                continue
            for tile_path in sorted(sample_dir.glob('c*_r*.png')):
                try:
                    col_part, row_part = tile_path.stem.split('_')
                    col = int(col_part[1:])
                    row = int(row_part[1:])
                except (ValueError, IndexError):
                    continue
                work.append(
                    {
                        'run': run_dir.name,
                        'sample': sample_dir.name,
                        'col': col,
                        'row': row,
                        'src_path': tile_path,
                        'scene_id': f'{run_dir.name}__{sample_dir.name}',
                        'tile_id': f'c{col}_r{row}',
                    }
                )
    return work


work = discover_tiles(SAMPLES_ROOT)
scenes = sorted({w['scene_id'] for w in work})
print(f'{len(work)} tiles across {len(scenes)} scenes')
if work:
    im = Image.open(work[0]['src_path'])
    print(f'first tile dims: {im.size[0]}x{im.size[1]} — {work[0]["src_path"].relative_to(REPO_ROOT)}')

450 tiles across 50 scenes
first tile dims: 1024x1024 — data/renderer/samples/seed42_n50_1779308635275/sample0/c-1_r-1.png


## 3. Provider — gpt-image-2 single-image edit

No mask, no fidelity flag (gpt-image-2 rejects it). Output is always 1024×1024.

In [5]:
MODEL_ID = 'gpt-image-2'
OUTPUT_SIZE = '1024x1024'

DEFAULT_PROMPT = (
    'Convert this aerial isometric city render into a 16-bit isometric pixel-art image '
    'in the style of SimCity 2000, RollerCoaster Tycoon 2, and Theme Hospital. Late-1990s '
    'simulation game aesthetic: limited saturated palette, crisp aliased pixel edges, '
    'simple flat shading with a single top-left light direction. Use the satellite content '
    'visible in the input as the strict source — every building, road, intersection, tree, '
    'water surface, and open lot in your output must correspond 1:1 to the input. Do not '
    'add, remove, relocate, resize, or invent anything. If unsure about a detail, prefer '
    'the input over your own priors. Treat low-poly artifacts as cues about real-world '
    'content, not features to copy: blocky tree shapes are trees (round pixel-art crowns), '
    'shimmering surfaces are water (flat color + 2-pixel checkerboard), stretched facades '
    'are buildings (clean rectangular pixel-art walls).'
)


def _openai_client():
    from openai import OpenAI

    key = os.environ.get('OPENAI_API_KEY')
    if not key:
        raise RuntimeError('OPENAI_API_KEY not set in .env')
    return OpenAI(api_key=key)


def stylize(image_bytes: bytes, prompt: str = DEFAULT_PROMPT) -> bytes:
    """Send PNG bytes to gpt-image-2 image-edit (no mask). Return PNG bytes."""
    client = _openai_client()
    resp = client.images.edit(
        model=MODEL_ID,
        image=('input.png', image_bytes, 'image/png'),
        prompt=prompt,
        size=OUTPUT_SIZE,
    )
    b64 = resp.data[0].b64_json
    if not b64:
        raise RuntimeError('OpenAI returned no image bytes')
    return base64.b64decode(b64)

## 4. Batch loop — parallel, resumable, live progress + spend

- **Parallel:** `ThreadPoolExecutor` with `MAX_PARALLEL` workers. Each tile is one independent API call; threads share nothing but the filesystem and per-call OpenAI clients.
- **Resumable:** tiles whose `target.png` already exists are filtered out before submission. Interrupt freely — re-running picks up where it left off.
- **Per-tile output:** `[n/N] ok   <scene>/<tile>  Xms  spent $Y.YY  rate Z/s  ETA Wmin`.
- **Cost knob:** `COST_PER_IMAGE_USD` — default `$0.042` (gpt-image-2 1024×1024 medium/auto, matches what the `stylize()` call uses since we don't pass `quality=`). The medium outputs from `exploration.ipynb` were already publishable quality, so this is the calibrated default. Bump to `0.167` if you switch to `quality='high'`.
- **Throughput knob:** `MAX_PARALLEL` — start at 8. Tier-1 OpenAI orgs may hit rate limits; raise to 16+ on higher tiers.

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

# --- tunables -------------------------------------------------------------
MAX_PARALLEL = 8            # workers in flight; OpenAI org RPM tier is the ceiling
COST_PER_IMAGE_USD = 0.042  # gpt-image-2 1024x1024 medium/auto — matches stylize() defaults
# --------------------------------------------------------------------------


def pair_dir(item: dict) -> Path:
    return TRAINING_ROOT / item['scene_id'] / item['tile_id']


def is_done(item: dict) -> bool:
    d = pair_dir(item)
    return (d / 'source.png').exists() and (d / 'target.png').exists() and (d / 'meta.json').exists()


def process_one(item: dict, prompt: str = DEFAULT_PROMPT) -> dict:
    d = pair_dir(item)
    d.mkdir(parents=True, exist_ok=True)

    src_bytes = item['src_path'].read_bytes()
    (d / 'source.png').write_bytes(src_bytes)

    t0 = time.time()
    target_bytes = stylize(src_bytes, prompt)
    dur_ms = int((time.time() - t0) * 1000)
    (d / 'target.png').write_bytes(target_bytes)

    src_im = Image.open(io.BytesIO(src_bytes))
    tgt_im = Image.open(io.BytesIO(target_bytes))
    meta = {
        'scene_id': item['scene_id'],
        'tile_id': item['tile_id'],
        'col': item['col'],
        'row': item['row'],
        'src_relpath': str(item['src_path'].relative_to(REPO_ROOT)),
        'src_size': list(src_im.size),
        'target_size': list(tgt_im.size),
        'model': MODEL_ID,
        'output_size': OUTPUT_SIZE,
        'prompt': prompt,
        'duration_ms': dur_ms,
        'generated_at': datetime.utcnow().isoformat() + 'Z',
    }
    (d / 'meta.json').write_text(json.dumps(meta, indent=2))
    return meta


todo = [w for w in work if not is_done(w)]
already = len(work) - len(todo)
print(f'{len(work)} total · {already} already done · {len(todo)} to process')
print(f'workers : {MAX_PARALLEL}')
print(f'est.    : ${len(todo) * COST_PER_IMAGE_USD:.2f}  ({len(todo)} × ${COST_PER_IMAGE_USD:.3f})')
print()

succeeded = 0
errors: list[dict] = []
lock = Lock()
start = time.time()

with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
    future_to_item = {ex.submit(process_one, item): item for item in todo}
    for n, fut in enumerate(as_completed(future_to_item), 1):
        item = future_to_item[fut]
        label = f'{item["scene_id"]}/{item["tile_id"]}'
        try:
            meta = fut.result()
            with lock:
                succeeded += 1
                elapsed = time.time() - start
                rate = succeeded / elapsed if elapsed > 0 else 0.0
                remaining = len(todo) - n
                eta_min = (remaining / rate / 60) if rate > 0 else float('inf')
                spent = succeeded * COST_PER_IMAGE_USD
                print(
                    f'[{n:>3}/{len(todo)}] ok   {label:55s} '
                    f'{meta["duration_ms"]:5d}ms  '
                    f'spent ${spent:6.2f}  rate {rate:.2f}/s  ETA {eta_min:5.1f}min'
                )
        except Exception as e:  # noqa: BLE001 — keep the pool alive past per-tile failures
            msg = f'{type(e).__name__}: {e}'
            with lock:
                errors.append({'tile': label, 'error': msg})
                print(f'[{n:>3}/{len(todo)}] FAIL {label:55s} {msg}')

elapsed = time.time() - start
print()
print(f'done in {elapsed / 60:.1f}min — {succeeded} succeeded, {len(errors)} failed')
print(f'total spent : ${succeeded * COST_PER_IMAGE_USD:.2f}')
if errors:
    print()
    print('failures (re-run this cell to retry — succeeded tiles are skipped):')
    for e in errors:
        print(f'  - {e["tile"]}: {e["error"]}')

450 total · 44 already done · 406 to process
workers : 8
est.    : $17.05  (406 × $0.042)



/var/folders/7t/f3k6qdvx067dct3mnxhlcwnw0000gn/T/ipykernel_87101/3700612238.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generated_at': datetime.utcnow().isoformat() + 'Z',


[  1/406] ok   seed42_n50_1779308635275__sample13/c0_r1                61071ms  spent $  0.04  rate 0.02/s  ETA 412.4min
[  2/406] ok   seed42_n50_1779308635275__sample13/c-1_r-1              63246ms  spent $  0.08  rate 0.03/s  ETA 213.0min
[  3/406] ok   seed42_n50_1779308635275__sample13/c1_r0                67764ms  spent $  0.13  rate 0.02/s  ETA 288.5min
[  4/406] ok   seed42_n50_1779308635275__sample13/c1_r1                69736ms  spent $  0.17  rate 0.03/s  ETA 222.8min
[  5/406] ok   seed42_n50_1779308635275__sample13/c0_r0                177526ms  spent $  0.21  rate 0.03/s  ETA 237.3min
[  6/406] ok   seed42_n50_1779308635275__sample12/c-1_r1               183568ms  spent $  0.25  rate 0.03/s  ETA 204.0min
[  7/406] ok   seed42_n50_1779308635275__sample13/c0_r-1               184698ms  spent $  0.29  rate 0.04/s  ETA 175.5min
[  8/406] ok   seed42_n50_1779308635275__sample13/c1_r-1               188316ms  spent $  0.34  rate 0.04/s  ETA 156.2min
[  9/406] ok   seed42_n50_17

## 5. Sanity check — random sample of completed pairs

Visual spot-check. Use this to eyeball quality before moving to the cull/augment stage.

In [ ]:
import random

import matplotlib.pyplot as plt

completed = [w for w in work if is_done(w)]
print(f'{len(completed)} completed pairs in {TRAINING_ROOT.relative_to(REPO_ROOT)}/')

N_PREVIEW = 6
sample = random.sample(completed, min(N_PREVIEW, len(completed)))

if sample:
    fig, axes = plt.subplots(len(sample), 2, figsize=(8, 4 * len(sample)))
    if len(sample) == 1:
        axes = [axes]
    for row, item in zip(axes, sample):
        d = pair_dir(item)
        row[0].imshow(Image.open(d / 'source.png'))
        row[0].set_title(f'source — {item["scene_id"]}/{item["tile_id"]}', fontsize=8)
        row[0].axis('off')
        row[1].imshow(Image.open(d / 'target.png'))
        row[1].set_title('target (gpt-image-2)', fontsize=8)
        row[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('no completed pairs yet — run the batch cell above first')